# IDS 570 – Text Extraction from Validated Dataset

This notebook:
1. Parses `Validated_dataset.docx` to extract categories, titles, source labels, descriptions, and URLs
2. Visits each URL (handles both PDFs and HTML pages, with a PubMed API shortcut)
3. Writes everything to a formatted Excel file

## Setup

In [1]:
# Run once to install dependencies
!pip install python-docx openpyxl requests beautifulsoup4 PyMuPDF lxml


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import fitz  # PyMuPDF
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import pandas as pd

In [ ]:
# -- Configuration -------------------------------------------------
DATA_DIR = Path("../01_data")
INPUT_DOCX = DATA_DIR / "Validated_dataset.docx"
OUTPUT_XLSX = DATA_DIR / "equity_corpus_scraped.xlsx"
OUTPUT_CONTEXT_CSV = DATA_DIR / "equity_contexts.csv"
REQUEST_DELAY = 2  # seconds between requests
REQUEST_TIMEOUT = 60  # seconds before giving up
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

## Step 1 – Parse the Word Document

We unzip the `.docx` and read the internal XML directly to reliably extract:
- Category headings
- Hyperlink URLs (from the relationships file)
- Link display text (title)
- Source labels (text after the em-dash)
- Description paragraphs (the line following each entry)

In [4]:
def parse_docx(path: str) -> list[dict]:
    """Extract entries from the docx by reading its internal XML."""
    W = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
    R = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
    PKG = "http://schemas.openxmlformats.org/package/2006/relationships"

    with zipfile.ZipFile(path) as z:
        rels_xml = ET.fromstring(z.read("word/_rels/document.xml.rels"))
        doc_xml = ET.fromstring(z.read("word/document.xml"))

    # rId → URL lookup
    url_map = {}
    for rel in rels_xml.findall(f"{{{PKG}}}Relationship"):
        if "hyperlink" in rel.get("Type", ""):
            url_map[rel.get("Id")] = rel.get("Target")

    body = doc_xml.find(f"{{{W}}}body")
    records = []
    current_category = "Uncategorized"

    for para in body.findall(f".//{{{W}}}p"):
        texts = [t.text or "" for t in para.iter(f"{{{W}}}t")]
        full_text = "".join(texts).strip()
        if not full_text:
            continue

        # Detect heading style
        pPr = para.find(f"{{{W}}}pPr")
        style_val = ""
        if pPr is not None:
            pStyle = pPr.find(f"{{{W}}}pStyle")
            if pStyle is not None:
                style_val = pStyle.get(f"{{{W}}}val", "")

        if re.match(r"(?i)heading\s*[12]", style_val):
            current_category = full_text
            continue

        # Look for hyperlinks
        hyperlinks = para.findall(f"{{{W}}}hyperlink")
        if not hyperlinks:
            if records and records[-1]["description"] == "":
                if not re.match(r"\*?\*?\d+\.?\*?\*?\s", full_text):
                    records[-1]["description"] = full_text
            continue

        hl = hyperlinks[0]
        rId = hl.get(f"{{{R}}}id")
        if not rId or rId not in url_map:
            continue
        url = url_map[rId]

        hl_texts = [t.text or "" for t in hl.iter(f"{{{W}}}t")]
        link_text = "".join(hl_texts).strip()

        source_label = ""
        if "---" in full_text or "\u2014" in full_text:
            source_label = re.sub(r".*?(---|\u2014)\s*", "", full_text).strip()
            source_label = source_label.strip("*").strip()

        records.append(
            {
                "category": current_category,
                "title": link_text,
                "source_label": source_label,
                "url": url,
                "description": "",
            }
        )

    return records

In [5]:
records = parse_docx(INPUT_DOCX)
print(f"Total entries extracted: {len(records)}\n")

# Show category breakdown
cats = {}
for r in records:
    cats[r["category"]] = cats.get(r["category"], 0) + 1
for cat, count in cats.items():
    print(f"  • {cat}: {count}")

Total entries extracted: 221

  • Federal policy / government health policy documents: 32
  • Academic article abstracts: 100
  • State or local health department reports/documents: 29
  • NGO / nonprofit health reports/documents: 30
  • Structural / justice framing: 7
  • Distribution / access / resource framing: 8
  • Policy implementation / measurement / systems framing: 9
  • Mixed / ambiguous framing: 6


In [6]:
# Quick preview of the first few entries
df_preview = pd.DataFrame(records)
df_preview.head(10)

,category,title,source_label,url,description
0,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,Federal methods report on defining and measuri...
1,Federal policy / government health policy docu...,HHS Action Plan to Reduce Racial and Ethnic He...,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,Department-wide action plan framed around disp...
2,Federal policy / government health policy docu...,"Agency Program, Activity, and Policy Highlight...",ASPE / HHS,https://aspe.hhs.gov/sites/default/files/docum...,Cross-agency highlights document tying program...
3,Federal policy / government health policy docu...,Advancing Equity through Quantitative Analysis,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/docum...,Technical report on quantitative approaches fo...
4,Federal policy / government health policy docu...,2023 Capacity Assessment Update,HHS,https://aspe.hhs.gov/sites/default/files/docum...,Cross-agency capacity assessment that referenc...
5,Federal policy / government health policy docu...,Landscape of Area-Level Deprivation Measures a...,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/docum...,Federal report on indices and methods used to ...
6,Federal policy / government health policy docu...,Building Data Capacity for Patient-Centered Ou...,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/2021-...,Data-capacity report linking evidence infrastr...
7,Federal policy / government health policy docu...,Health Equity Guide,CDC,https://www.cdc.gov/health-communication/pdf/h...,Brief CDC guide defining health equity and com...
8,Federal policy / government health policy docu...,Focusing on Health Equity,CDC,https://stacks.cdc.gov/view/cdc/126226/cdc_126...,CDC overview document on the meaning and opera...
9,Federal policy / government health policy docu...,Promoting Health Equity: A Resource to Help Co...,CDC,https://stacks.cdc.gov/view/cdc/11130/cdc_1113...,Foundational CDC resource linking equity to SD...


## Step 2 – Scrape Text from Each URL

Three strategies:
- **PubMed links** → Use the NCBI E-utilities API (reliable, no JavaScript issues)
- **PDF links** → Download in memory and extract with PyMuPDF
- **HTML pages** → Parse with BeautifulSoup using targeted selectors

In [7]:
def extract_pmid(url: str) -> str | None:
    """Extract PubMed ID from a URL."""
    m = re.search(r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)", url)
    return m.group(1) if m else None


def scrape_pubmed_api(pmid: str) -> str:
    """Fetch abstract via NCBI E-utilities."""
    efetch_url = (
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        f"?db=pubmed&id={pmid}&rettype=abstract&retmode=text"
    )
    try:
        resp = requests.get(efetch_url, timeout=30)
        resp.raise_for_status()
        return resp.text.strip()
    except Exception as e:
        return f"ERROR (PubMed API): {e}"


def scrape_url(url: str) -> str:
    """Download and extract text from a PDF or HTML page."""
    try:
        resp = requests.get(
            url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True
        )
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")

        # ── PDF ──
        if "pdf" in content_type.lower() or url.lower().rstrip("/").endswith(".pdf"):
            doc = fitz.open(stream=resp.content, filetype="pdf")
            text = "\n".join(page.get_text() for page in doc)
            doc.close()
            return text.strip()

        # ── HTML ──
        soup = BeautifulSoup(resp.text, "lxml")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        selectors = [
            "#abstract",
            ".abstract-content",
            "#enc-abstract",
            "div.abstract",
            "article",
            "main",
            '[role="main"]',
            "#content",
            ".content",
        ]
        for sel in selectors:
            node = soup.select_one(sel)
            if node and len(node.get_text(strip=True)) > 100:
                return node.get_text(separator="\n", strip=True)

        body = soup.find("body")
        if body:
            return body.get_text(separator="\n", strip=True)
        return soup.get_text(separator="\n", strip=True)

    except Exception as e:
        return f"ERROR: {e}"

In [8]:
# ── Scrape all URLs ──────────────────────────────────────
# This takes ~8-10 min for ~225 URLs with a 2s delay.

total = len(records)
print(f"Scraping {total} URLs (estimated {total * REQUEST_DELAY // 60} min)...\n")

for i, rec in enumerate(records, 1):
    url = rec["url"]
    short_url = url[:80] + ("..." if len(url) > 80 else "")

    pmid = extract_pmid(url)
    if pmid:
        rec["scraped_text"] = scrape_pubmed_api(pmid)
    else:
        rec["scraped_text"] = scrape_url(url)

    status = "✓" if not rec["scraped_text"].startswith("ERROR") else "✗"
    chars = len(rec["scraped_text"])
    print(f"[{i}/{total}] {status} {chars:>7,} chars │ {short_url}")

    time.sleep(REQUEST_DELAY)

n_ok = sum(1 for r in records if not r["scraped_text"].startswith("ERROR"))
n_fail = total - n_ok
print(f"\n{'='*50}")
print(f"Done! {n_ok} succeeded, {n_fail} failed")

Scraping 221 URLs (estimated 7 min)...

[1/221] ✓ 209,287 chars │ https://aspe.hhs.gov/sites/default/files/private/pdf/265566/developing-health-eq...
[2/221] ✓ 101,694 chars │ https://aspe.hhs.gov/sites/default/files/private/pdf/206166/DisparitiesActionPla...
[3/221] ✓ 187,896 chars │ https://aspe.hhs.gov/sites/default/files/documents/514310c76b9ed8afa49cd223002b4...
[4/221] ✓  36,744 chars │ https://aspe.hhs.gov/sites/default/files/documents/41fbd5e6cbc181b77f96076c3ec70...
[5/221] ✓  59,950 chars │ https://aspe.hhs.gov/sites/default/files/documents/fbbe2c601e231a8ce9089e4f02e27...
[6/221] ✓ 281,139 chars │ https://aspe.hhs.gov/sites/default/files/documents/8dc674c904723bf8a5ce4cfc8d3dc...
[7/221] ✓  98,871 chars │ https://aspe.hhs.gov/sites/default/files/2021-07/covid-pcor-report.pdf
[8/221] ✓  10,239 chars │ https://www.cdc.gov/health-communication/pdf/health-equity-guide-infographic-508...
[9/221] ✗     101 chars │ https://stacks.cdc.gov/view/cdc/126226/cdc_126226_DS1.pdf
[10/221] 

## Step 3 – Preview Results

In [ ]:
df = pd.DataFrame(records)
df["text_length"] = df["scraped_text"].str.len()
df["is_error"] = df["scraped_text"].str.startswith("ERROR")

summary = (
    df.groupby("category")
    .agg(
        entries=("url", "count"),
        success=("is_error", lambda x: (~x).sum()),
        failed=("is_error", "sum"),
        avg_chars=("text_length", lambda x: x[~df.loc[x.index, "is_error"]].mean()),
    )
    .round(0)
)

summary

,entries,success,failed,avg_chars
category,,,,
Academic article abstracts,100,92,8,4455.0
Distribution / access / resource framing,8,6,2,16679.0
Federal policy / government health policy documents,32,27,5,71030.0
Mixed / ambiguous framing,6,6,0,15091.0
NGO / nonprofit health reports/documents,30,30,0,51227.0
Policy implementation / measurement / systems framing,9,5,4,15293.0
State or local health department reports/documents,29,26,3,116756.0
Structural / justice framing,7,7,0,12336.0


In [10]:
# Show any failures for manual inspection
failures = df[df["is_error"]][["category", "title", "url", "scraped_text"]]
if len(failures) > 0:
    print(f"{len(failures)} failed URLs:\n")
    for _, row in failures.iterrows():
        print(f"  • {row['title']}")
        print(f"    {row['url']}")
        print(f"    {row['scraped_text'][:100]}\n")
else:
    print("All URLs scraped successfully!")

22 failed URLs:

  • Focusing on Health Equity
    https://stacks.cdc.gov/view/cdc/126226/cdc_126226_DS1.pdf
    ERROR: 403 Client Error: Forbidden for url: https://stacks.cdc.gov/view/cdc/126226/cdc_126226_DS1.pd

  • Promoting Health Equity: A Resource to Help Communities Address Social Determinants of Health
    https://stacks.cdc.gov/view/cdc/11130/cdc_11130_DS1.pdf
    ERROR: 403 Client Error: Forbidden for url: https://stacks.cdc.gov/view/cdc/11130/cdc_11130_DS1.pdf

  • Health Equity Report 2019–2020
    https://www.hrsa.gov/sites/default/files/hrsa/about/news/hrsa-health-equity-report.pdf
    ERROR: 403 Client Error: Forbidden for url: https://www.hrsa.gov/sites/default/files/hrsa/about/news

  • Healthy People 2030: Health Equity in Healthy People 2030
    https://health.gov/healthypeople
    ERROR: 404 Client Error: Not Found for url: https://health.gov/healthypeople

  • SAMHSA Behavioral Health Equity Framework
    https://www.samhsa.gov/behavioral-health-equity
    ERROR: 

## Step 4 – Write to Excel

In [12]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


def sanitize_excel_value(value):
    if not isinstance(value, str):
        return value
    return ILLEGAL_CHARACTERS_RE.sub("", value)


def write_excel(records: list[dict], output_path: str):
    wb = Workbook()
    ws = wb.active
    ws.title = "Corpus"

    columns = [
        ("Entry", 8),
        ("Category", 35),
        ("Title", 50),
        ("Source", 20),
        ("URL", 60),
        ("Description", 50),
        ("Scraped_Text", 80),
    ]

    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill("solid", fgColor="2F5496")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin_border = Border(bottom=Side(style="thin", color="B0B0B0"))
    cell_font = Font(name="Arial", size=10)
    wrap_align = Alignment(vertical="top", wrap_text=True)

    for col_idx, (name, width) in enumerate(columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_align
        ws.column_dimensions[cell.column_letter].width = width

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:G{len(records) + 1}"

    for row_idx, rec in enumerate(records, 2):
        values = [
            row_idx - 1,
            rec["category"],
            rec["title"],
            rec["source_label"],
            rec["url"],
            rec["description"],
            rec["scraped_text"],
        ]
        for col_idx, val in enumerate(values, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=sanitize_excel_value(val))
            cell.font = cell_font
            cell.alignment = wrap_align
            cell.border = thin_border

    wb.save(output_path)
    print(f"Wrote {len(records)} rows -> {output_path}")


write_excel(records, OUTPUT_XLSX)

Wrote 221 rows -> equity_corpus_scraped.xlsx


## Step 5 – Extract "Equity" Contexts

Extract ±1 sentence around every occurrence of "equity".


In [9]:
import re


SENTENCE_BOUNDARY_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text or text.startswith("ERROR"):
        return []
    return [
        sentence.strip()
        for sentence in SENTENCE_BOUNDARY_RE.split(text)
        if sentence.strip()
    ]


def extract_equity_contexts(text: str, window: int = 1) -> list[str]:
    """Return +/-window sentences around every sentence containing 'equity'."""
    sentences = split_sentences(text)
    contexts = []
    for i, sent in enumerate(sentences):
        if re.search(r"\bequity\b", sent, re.IGNORECASE):
            start = max(0, i - window)
            end = min(len(sentences), i + window + 1)
            context = " ".join(sentences[start:end])
            contexts.append(context)
    return contexts


context_rows = []
for rec in records:
    contexts = extract_equity_contexts(rec["scraped_text"])
    for ctx in contexts:
        context_rows.append(
            {
                "category": rec["category"],
                "title": rec["title"],
                "source_label": rec["source_label"],
                "url": rec["url"],
                "equity_context": ctx,
            }
        )

df_contexts = pd.DataFrame(context_rows)
print(f"Extracted {len(df_contexts)} equity contexts from {len(records)} documents")
df_contexts.head(10)

Extracted 6109 equity contexts from 221 documents


,category,title,source_label,url,equity_context
0,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,C O N T R A C T O R P R O J E C T R E P O R T ...
1,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,"ASPE wrote two Reports to Congress, making rec..."
2,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,This included the recommendations that the Cen...
3,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,"Moreover, in the ASPE commissioned report, Sys..."
4,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,"In response to this challenge, ASPE asked the ..."
5,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,RAND identified 10 existing approaches to heal...
6,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,"Based on input from RAND, ASPE, and the TEP, i..."
7,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,The purpose of including health equity measure...
8,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,It could also encourage providers to improve h...
9,Federal policy / government health policy docu...,Developing Health Equity Measures,ASPE / HHS,https://aspe.hhs.gov/sites/default/files/priva...,"Washington, DC: The National Academies Press. ..."


In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


def sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    object_columns = cleaned.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        cleaned[column] = cleaned[column].map(
            lambda value: (
                ILLEGAL_CHARACTERS_RE.sub("", value)
                if isinstance(value, str)
                else value
            )
        )
    return cleaned


OUTPUT_CONTEXT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_contexts_excel = sanitize_dataframe_for_excel(df_contexts)
df_contexts_excel.to_csv(OUTPUT_CONTEXT_CSV, index=False)
print(f"Saved -> {OUTPUT_CONTEXT_CSV}")

Saved -> equity_contexts.xlsx
